In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys

# Suppress all warnings
import warnings

warnings.filterwarnings("ignore")  # temporarilty to fix output problem
sys.path.append("../../existing_metric_comparison/")
sys.path.append("../../../")
sys.path.append("../../utils/HLCA")
import data_cleanup

# import metrics_lib as metrics
import os
import mapqc

In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

## Set global variables

In [ ]:
testrun = False

choose query condition to subset to:

In [ ]:
condition = "IPF"  # "IPF" or "Healthy" (showed Healthy in paper)

set mapqc params:

In [ ]:
k = 500
max_k = 5000
n_nhoods = 500

In [ ]:
study = "Sheppard_2020"
batch_variable = "sample"
kl = "1.0"
use_old_counts = True
if use_old_counts == True:
    suffix = "original_counts_"
else:
    suffix = "cellxgene_counts_"

path_emb = f"/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HLCA/HLCA_and_{study}_{suffix}batch_{batch_variable}_kl{kl}_latent.h5ad"
path_counts = f"/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HLCA/HLCA_and_{study}_counts.h5ad"
path_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/mapqc/results/HLCA/{study}/{suffix}batch_{batch_variable}_kl{kl}/benchmark"
path_mapqc_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/mapqc/results/HLCA/{study}/{suffix}batch_{batch_variable}_kl{kl}/mapqc"

In [ ]:
output_file_path_prefix = f"/lustre/groups/ml01/code/lisa.sikkema/mapqc/results/downsampling_query/{study}_{suffix}batch_{batch_variable}_kl{kl}_"

In [ ]:
output_file_path_prefix

In [ ]:
if not os.path.exists(path_results_dir):
    if os.path.exists(os.path.dirname(path_results_dir)):
        print(f"Creating directory {path_results_dir}")
        os.mkdir(RESULT_DIR)
    else:
        print("Parent directory of path_results_dir does not exist. Create it first.")

In [ ]:
path_emb = "/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HLCA/HLCA_and_Sheppard_2020_original_counts_batch_sample_kl1.0_latent.h5ad"

In [ ]:
path_counts = "/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HLCA/HLCA_and_Sheppard_2020_counts.h5ad"

In [ ]:
n_repeats = 2

In [ ]:
path_results_dir

## load and prep data

In [ ]:
adata_emb = sc.read_h5ad(path_emb)
adata_counts = sc.read_h5ad(path_counts)
adata = data_cleanup.add_mapping_info_to_counts_adata(adata_counts, adata_emb)

subset to condition of interest:

In [ ]:
adata = adata[
    (adata.obs.r_or_q == "r") | (adata.obs.lung_condition == condition), :
].copy()

In [ ]:
adata.shape

In [ ]:
if testrun:
    # subset to appr. 5% of data by subsampling samples
    ref_samples = adata.obs.loc[adata.obs.r_or_q == "r", "sample"].unique()
    q_samples = adata.obs.loc[adata.obs.r_or_q == "q", "sample"].unique()
    random_subset_ref = np.random.choice(
        ref_samples,
        size=int(len(ref_samples) / 20),
        replace=False,
    )  # np.arange(0,adata.n_obs)
    random_subset_q = np.random.choice(
        q_samples,
        size=int(len(q_samples) / 2),
        replace=False,
    )
    sample_subset = list(random_subset_ref) + list(random_subset_q)
    adata = adata[adata.obs["sample"].isin(sample_subset), :].copy()
    print(
        "Number of query samples after subsampling:",
        adata.obs.loc[adata.obs.r_or_q == "q", "sample"].nunique(),
    )
    # set mapqc n nhoods to really low for minimal computation time
    n_nhoods = 5

In [ ]:
sc.pl.umap(adata, color="leiden", legend_loc="on data")

In [ ]:
adata.shape

## Run metrics with downsampling of query

In [ ]:
downsample_fractions = [1, 0.1, 0.01]

In [ ]:
query_cells = adata.obs.index[adata.obs.r_or_q == "q"].tolist()

In [ ]:
lisi_batch_results = adata.obs.loc[
    query_cells, [f"ann_level_{lev}_pred" for lev in range(1, 6)]
].copy()
mapqc_results_edistance = adata.obs.loc[
    query_cells, [f"ann_level_{lev}_pred" for lev in range(1, 6)]
].copy()
mapqc_results_pw_euclidean = adata.obs.loc[
    query_cells, [f"ann_level_{lev}_pred" for lev in range(1, 6)]
].copy()

In [ ]:
for frac in downsample_fractions:
    print("Working on downsample fraction", frac)
    for cycle in range(0, n_repeats):
        query_subset = np.random.choice(
            query_cells, size=int(frac * len(query_cells)), replace=False
        )
        adata_subset = adata[
            (adata.obs.r_or_q == "r") | adata.obs.index.isin(query_subset), :
        ].copy()
        query_subset_ordered = adata_subset.obs.index[
            adata_subset.obs.index.isin(query_subset)
        ]
        # start with lisi
        lisi_batch_results[f"{frac} ({cycle})"] = np.nan
        lisi_batch_results.loc[
            query_subset_ordered, f"{frac} ({cycle})"
        ] = metrics.calculate_scib_metrics(
            adata=adata_subset,
            obsm_emb_key="X_scanvi_emb",
            batch_key="r_or_q",
            dataset_key="r_or_q",
            sample_key="sample",
            dry_run=False,
        )[
            "lisi_batch"
        ][
            "per_cell"
        ].loc[
            query_subset_ordered
        ]
        # run edistance mapqc
        mapqc_results_downsample = mapqc.MapqcResults(
            adata=adata_subset,
            ref_q_key="r_or_q",
            ref_cat="r",
            query_cat="q",
            condition_key="status",
            control_cats="control",
            control_or_case="case",
            case_cats=["IPF"],
            emb_key="X_scanvi_emb",
            dataset_key="study",
            sample_key="sample",
            sampling_group_cat="leiden",
            n_cores=4,
            distance_type="edistance",
            verbose=False,
            k=k,
            max_k=max_k,
            n_nhoods=n_nhoods,
        )
        mapqc_results_downsample.run_mapqc()
        mapqc_results_edistance[f"{frac} ({cycle})"] = np.nan
        mapqc_results_edistance.loc[
            query_subset_ordered, f"{frac} ({cycle})"
        ] = mapqc_results_downsample.adata.obs.loc[query_subset_ordered, "mapqc_score"]

        # now run pairwise euclidean mapqc
        mapqc_results_downsample = mapqc.MapqcResults(
            adata=adata_subset,
            ref_q_key="r_or_q",
            ref_cat="r",
            query_cat="q",
            condition_key="status",
            control_cats="control",
            control_or_case="case",
            case_cats=["IPF"],
            emb_key="X_scanvi_emb",
            dataset_key="study",
            sample_key="sample",
            sampling_group_cat="leiden",
            n_cores=4,
            verbose=False,
            k=k,
            max_k=max_k,
            n_nhoods=n_nhoods,
        )
        mapqc_results_downsample.run_mapqc()
        mapqc_results_pw_euclidean[f"{frac} ({cycle})"] = np.nan
        mapqc_results_pw_euclidean.loc[
            query_subset_ordered, f"{frac} ({cycle})"
        ] = mapqc_results_downsample.adata.obs.loc[query_subset_ordered, "mapqc_score"]
        lisi_batch_results.to_csv(
            os.path.join(path_results_dir, f"{condition}_all_cells_lisi.csv")
        )
        mapqc_results_edistance.to_csv(
            os.path.join(path_results_dir, f"{condition}_all_cells_mapqc_edistance.csv")
        )
        mapqc_results_pw_euclidean.to_csv(
            os.path.join(
                path_results_dir, f"{condition}_all_cells_mapqc_pw_euclidean.csv"
            )
        )
        del adata_subset, mapqc_results_downsample

store results

In [ ]:
path_results_dir

load results if starting from here:

In [ ]:
if condition == "Healthy":
    prefix = "ctr"
elif condition == "IPF":
    prefix = "IPF"
lisi_batch_results = pd.read_csv(
    os.path.join(path_results_dir, f"{prefix}_all_cells_lisi.csv"), index_col=0
)
mapqc_results_edistance = pd.read_csv(
    os.path.join(path_results_dir, f"{prefix}_all_cells_mapqc_edistance.csv"),
    index_col=0,
)
mapqc_results_pw_euclidean = pd.read_csv(
    os.path.join(path_results_dir, f"{prefix}_all_cells_mapqc_pw_euclidean.csv"),
    index_col=0,
)

if results were in two parts (due to kernel dying in between, this was the case for the IPF run):

In [ ]:
lisi_batch_results_part2 = pd.read_csv(
    os.path.join(path_results_dir, f"{condition}_all_cells_lisi_part2.csv"), index_col=0
)
mapqc_results_edistance_part2 = pd.read_csv(
    os.path.join(path_results_dir, f"{condition}_all_cells_mapqc_edistance_part2.csv"),
    index_col=0,
)
mapqc_results_pw_euclidean_part2 = pd.read_csv(
    os.path.join(
        path_results_dir, f"{condition}_all_cells_mapqc_pw_euclidean_part2.csv"
    ),
    index_col=0,
)

In [ ]:
for col in lisi_batch_results_part2.columns:
    if col not in lisi_batch_results.columns:
        lisi_batch_results[col] = lisi_batch_results_part2[col]
        mapqc_results_edistance[col] = mapqc_results_edistance_part2[col]
        mapqc_results_pw_euclidean[col] = mapqc_results_pw_euclidean_part2[col]

## Plot results of different runs

correct Pericyte annotations:

In [ ]:
ref_col = "ann_level_5_pred"
for lev in [2, 3]:
    ann_colname = f"ann_level_{lev}_pred"
    # mapqc pw euclidean
    mapqc_results_pw_euclidean[ann_colname] = mapqc_results_pw_euclidean[
        ann_colname
    ].tolist()
    mapqc_results_pw_euclidean.loc[
        mapqc_results_pw_euclidean[ref_col] == "Pericytes",
        ann_colname,
    ] = "Pericytes"
    # mapqc energy dist
    mapqc_results_edistance[ann_colname] = mapqc_results_edistance[ann_colname].tolist()
    mapqc_results_edistance.loc[
        mapqc_results_edistance[ref_col] == "Pericytes",
        ann_colname,
    ] = "Pericytes"
    # lisi batch
    lisi_batch_results[ann_colname] = lisi_batch_results[ann_colname].tolist()
    lisi_batch_results.loc[
        lisi_batch_results[ref_col] == "Pericytes",
        ann_colname,
    ] = "Pericytes"

### batch lisi:

In [ ]:
results_df_lisi = pd.DataFrame()

In [ ]:
for frac in downsample_fractions:
    for cycle in range(0, n_repeats):
        results_df_lisi = pd.concat(
            (
                results_df_lisi,
                pd.DataFrame(
                    {
                        "lisi_score": lisi_batch_results[f"{frac} ({cycle})"],
                        "run": f"downsample {frac} ({cycle})",
                        "ann_level_3_pred": lisi_batch_results.ann_level_3_pred,
                    }
                ),
            )
        )
results_df_lisi.dropna(how="any", inplace=True)

as lisi has results for all cells, we will use the lisi results to determine how many cells of each cell type were present after downsampling for each run:

In [ ]:
cellcount_per_ct_per_run = pd.crosstab(
    results_df_lisi.ann_level_3_pred, results_df_lisi.run
)

now we can filter our results such that only results for cell types with a minimum of min_n_cells per cell type are shown. We do not expect stable results for lower cell numbers anyway.

In [ ]:
min_n_cells = 50
ct_per_run_to_mask = list()
for run in cellcount_per_ct_per_run.columns:
    for ct in cellcount_per_ct_per_run.index:
        if cellcount_per_ct_per_run.loc[ct, run] < min_n_cells:
            ct_per_run_to_mask.append((run, ct))

In [ ]:
cellcount_per_ct_per_run_masked = cellcount_per_ct_per_run[
    cellcount_per_ct_per_run > min_n_cells
]  # will be used for xtick labels further down

now we can filter the results accordingly:

In [ ]:
results_df_lisi["filt"] = True
for run, ct in ct_per_run_to_mask:
    results_df_lisi.loc[
        (results_df_lisi.run == run) & (results_df_lisi.ann_level_3_pred == ct),
        "filt",
    ] = False

In [ ]:
results_df_lisi_filt = results_df_lisi.loc[results_df_lisi.filt, :].copy()

set color palette for figures:

In [ ]:
palette = {
    "downsample 1 (0)": "tab:blue",
    "downsample 1 (1)": "tab:blue",
    "downsample 0.1 (0)": "tab:green",
    "downsample 0.1 (1)": "tab:green",
    "downsample 0.01 (0)": "tab:purple",
    "downsample 0.01 (1)": "tab:purple",
}

In [ ]:
fig, ax = plt.subplots(figsize=(20, 2))
sns.boxplot(
    results_df_lisi_filt,
    x="ann_level_3_pred",
    y="lisi_score",
    hue="run",
    order=sorted(results_df_lisi_filt.ann_level_3_pred.unique()),
    ax=ax,
    fliersize=0,
    palette=palette,
)
# sns.stripplot(
#     results_df_lisi_filt,
#     x="ann_level_3_pred",
#     y="lisi_score",
#     hue="run",
#     order=sorted(results_df_lisi_filt.ann_level_3_pred.unique()),
#     hue_
#     ax=ax,
#     size=1,
#     color="black",
# )
ax.tick_params(axis="x", rotation=90)
x_tick_labels = [label.get_text() for label in ax.get_xticklabels()]
x_tick_labels_new = list()
for label in x_tick_labels:
    suffix = str(cellcount_per_ct_per_run_masked.loc[label, :][-1].astype(int))
    x_tick_labels_new.append(f"{label} (n={suffix})")
ax.set_xticklabels(x_tick_labels_new)

leg = plt.legend(loc=(1.01, 0), frameon=False, title="Downsample fraction (run number)")
leg._legend_box.align = "left"
ax.set_title(f"Results LISI ({condition})")
ax.set_xlabel("Cell type (n cells in full data)")
ax.set_ylabel("Batch LISI score")
plt.show()

#### mapqc pairwise euclidean:

In [ ]:
results_df_mapqc_pw = pd.DataFrame()

In [ ]:
for frac in downsample_fractions:
    for cycle in range(0, n_repeats):
        results_df_mapqc_pw = pd.concat(
            (
                results_df_mapqc_pw,
                pd.DataFrame(
                    {
                        "mapqc_score": mapqc_results_pw_euclidean[f"{frac} ({cycle})"],
                        "run": f"downsample {frac} ({cycle})",
                        "ann_level_3_pred": mapqc_results_pw_euclidean.ann_level_3_pred,
                    }
                ),
            )
        )
results_df_mapqc_pw.dropna(how="any", inplace=True)

In [ ]:
results_df_mapqc_pw.shape

In [ ]:
results_df_mapqc_pw["filt"] = True
for run, ct in ct_per_run_to_mask:
    results_df_mapqc_pw.loc[
        (results_df_mapqc_pw.run == run) & (results_df_mapqc_pw.ann_level_3_pred == ct),
        "filt",
    ] = False

In [ ]:
results_df_mapqc_pw_filt = results_df_mapqc_pw.loc[results_df_mapqc_pw.filt, :].copy()

In [ ]:
fig, ax = plt.subplots(figsize=(20, 3))
sns.boxplot(
    results_df_mapqc_pw_filt,
    x="ann_level_3_pred",
    y="mapqc_score",
    hue="run",
    order=sorted(results_df_mapqc_pw_filt.ann_level_3_pred.unique()),
    ax=ax,
    fliersize=0,
    palette=palette,
)
plt.axhline(y=2, color="red", linestyle="--")
leg = plt.legend(loc=(1.01, 0), frameon=False, title="Downsample fraction (run number)")
leg._legend_box.align = "left"
x_tick_labels = [label.get_text() for label in ax.get_xticklabels()]
x_tick_labels_new = list()
for label in x_tick_labels:
    suffix = str(cellcount_per_ct_per_run_masked.loc[label, :][-1].astype(int))
    x_tick_labels_new.append(f"{label} (n={suffix})")
ax.set_xticklabels(x_tick_labels_new)
# sns.stripplot(results_df_lisi, x="run", y="lisi_score", ax=ax, size=0.1, color="black")
ax.tick_params(axis="x", rotation=90)
ax.set_ylim((-5, 15))
ax.set_title(f"Results pairwise euclidean distance ({condition}, k={k})")
ax.set_xlabel("Cell type (n cells in full data)")
ax.set_ylabel("MapQC score")
plt.show()

#### mapqc results e-distance:

In [ ]:
results_df_mapqc_edistance = pd.DataFrame()

In [ ]:
for frac in downsample_fractions:
    for cycle in range(0, n_repeats):
        results_df_mapqc_edistance = pd.concat(
            (
                results_df_mapqc_edistance,
                pd.DataFrame(
                    {
                        "mapqc_score": mapqc_results_edistance[f"{frac} ({cycle})"],
                        "run": f"downsample {frac} ({cycle})",
                        "ann_level_3_pred": mapqc_results_edistance.ann_level_3_pred,
                    }
                ),
            )
        )
results_df_mapqc_edistance.dropna(how="any", inplace=True)

In [ ]:
results_df_mapqc_edistance.shape

In [ ]:
results_df_mapqc_edistance["filt"] = True
for run, ct in ct_per_run_to_mask:
    results_df_mapqc_edistance.loc[
        (results_df_mapqc_edistance.run == run)
        & (results_df_mapqc_edistance.ann_level_3_pred == ct),
        "filt",
    ] = False

In [ ]:
results_df_mapqc_edistance_filt = results_df_mapqc_edistance.loc[
    results_df_mapqc_edistance.filt, :
].copy()

In [ ]:
fig, ax = plt.subplots(figsize=(20, 3))
sns.boxplot(
    results_df_mapqc_edistance_filt,
    x="ann_level_3_pred",
    y="mapqc_score",
    order=sorted(results_df_mapqc_edistance_filt.ann_level_3_pred.unique()),
    hue="run",
    ax=ax,
    fliersize=0,
    palette=palette,
)
# sns.stripplot(results_df_lisi, x="run", y="lisi_score", ax=ax, size=0.1, color="black")
plt.axhline(y=2, color="red", linestyle="--")
ax.tick_params(axis="x", rotation=90)
ax.set_ylim((-5, 15))
x_tick_labels = [label.get_text() for label in ax.get_xticklabels()]
x_tick_labels_new = list()
for label in x_tick_labels:
    suffix = str(cellcount_per_ct_per_run_masked.loc[label, :][-1].astype(int))
    x_tick_labels_new.append(f"{label} (n={suffix})")
ax.set_xticklabels(x_tick_labels_new)
leg = plt.legend(loc=(1.01, 0), frameon=False, title="Downsample fraction (run number)")
leg._legend_box.align = "left"
ax.set_title(f"Results energy distance ({condition}, k={k})")
ax.set_ylabel("MapQC score")
ax.set_xlabel("Cell type (n cells in full data)")
plt.show()